# Notebook Setup

In [ ]:
!pip install polars==1.33
!pip install plotnine

In [ ]:
import os
import glob
import subprocess

import polars as pl
from scipy.stats import false_discovery_control
from statsmodels.stats.multitest import multipletests
from plotnine import *

from datetime import datetime
# Record time needed to run the notebook
start = datetime.now()

In [ ]:
# Initialize Hail without setting default reference
import hail as hl
hl.init()

# Set the default reference genome after initialization
hl.default_reference('GRCh38')

In [ ]:
#Path functions to make it easier to load data tables
def get_ht_path(type, ancestry, pheno):
    return f'gs://vwb-aou-allxall/v8/ht/{type}/{ancestry.upper()}/phenotype_{pheno}_{type}_results.ht'

def get_mt_path(ancestry, type):
    return f'gs://vwb-aou-allxall/v8/mt/{ancestry.upper()}_{type}_results.mt'

# Extract Variant metadata

In [ ]:
OUT_PATH = os.environ["AOU_GYM_DATA_ROOT"].rstrip("/") + "/allxall_exome_sumstats"

# gene_list = list(gt_df.filter(pl.col('Pvalue_FDR')<0.05)['gene_id'].unique())
# print("Number of unique genes:", len(gene_list))

ht = hl.read_table('gs://vwb-aou-allxall/v8/utility_ht/aou_exome_variant_qc_annotated.ht')

# Filter to your genes (top-level gene_id — easy)
# ht = ht.filter(hl.literal(set(gene_list)).contains(ht.gene_id))

wc = ht.vep.worst_csq_by_gene_canonical   # gene-specific, canonical/MANE transcript

ht = ht.annotate(
    variant_id = hl.delimit([
        ht.locus.contig, hl.str(ht.locus.position),
        ht.alleles[0], ht.alleles[1]
    ], ':'),
    chrom = ht.locus.contig,
    pos   = ht.locus.position,
    ref   = ht.alleles[0],
    alt   = ht.alleles[1],
    # gene-specific canonical consequence
    vep_consequence     = wc.most_severe_consequence,
    consequence_terms   = hl.delimit(wc.consequence_terms, ','),
    impact              = wc.impact,
    transcript_id       = wc.transcript_id,
    mane_select         = wc.mane_select,
    canonical           = wc.canonical,
    lof                 = wc.lof,
    lof_filter          = wc.lof_filter,
    lof_flags           = wc.lof_flags,
    polyphen_prediction = wc.polyphen_prediction,
    polyphen_score      = wc.polyphen_score,
    sift_prediction     = wc.sift_prediction,
    sift_score          = wc.sift_score,
    amino_acids         = wc.amino_acids,
    codons              = wc.codons,
    hgvsc               = wc.hgvsc,
    hgvsp               = wc.hgvsp,
    biotype             = wc.biotype,
    protein_id          = wc.protein_id,
    cds_start           = wc.cds_start,
    cds_end             = wc.cds_end,
    protein_start       = wc.protein_start,
    protein_end         = wc.protein_end,
    csq_score           = wc.csq_score,
    # per-ancestry allele frequencies (alt allele = index 1)
    AF_AFR = ht.freq.AFR.AF[1],
    AF_AMR = ht.freq.AMR.AF[1],
    AF_EAS = ht.freq.EAS.AF[1],
    AF_EUR = ht.freq.EUR.AF[1],
    AF_MID = ht.freq.MID.AF[1],
    AF_SAS = ht.freq.SAS.AF[1],
    AF_ALL = ht.freq.ALL.AF[1],
    AC_ALL = ht.freq.ALL.AC[1],
)

ht = ht.key_by()
ht = ht.select(
    'variant_id', 'chrom', 'pos', 'ref', 'alt',
    'gene_id', 'gene_symbol', 'annotation',
    'vep_consequence', 'consequence_terms', 'impact',
    'transcript_id', 'mane_select', 'canonical',
    'lof', 'lof_filter', 'lof_flags',
    'polyphen_prediction', 'polyphen_score',
    'sift_prediction', 'sift_score',
    'amino_acids', 'codons', 'hgvsp', 'hgvsc',
    'biotype', 'protein_start', 'protein_id',
    'AF_AFR', 'AF_AMR', 'AF_EAS', 'AF_EUR', 'AF_MID', 'AF_SAS', 'AF_ALL', 'AC_ALL',
)

# Write distributedly to GCS
ht.to_spark().write.mode('overwrite').parquet(f'{OUT_PATH}/aou_exome_variant_qc_annotated.parquet')